# Análisis de la Brecha de Ingresos entre Zonas Urbanas y Rurales
Este análisis utiliza los datos de la encuesta Casen 2024 para visualizar cómo evoluciona la brecha de ingresos laborales entre las zonas urbanas y rurales a lo largo del ciclo vital (edad). 

Se seleccionan las siguientes variables según el Libro de Códigos:
- `activ` == 1: Filtra solo a las personas **ocupadas**.
- `yoprcor`: Mide el **ingreso de la ocupación principal**, que refleja con mayor precisión la compensación directa por el trabajo.
- `edad`: Permite usar un valor continuo para visualizar la evolución temporal en formato de línea.
- `area`: Mide la **zona geográfica** (1 = Urbano, 2 = Rural).


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Carga y preparación de los datos
df = pd.read_parquet('../Data/casen_2024.parquet')

# Filtramos personas ocupadas (activ == 1.0) con ingresos de ocupación principal mayores a 0
# Además acotamos la edad a un rango laboral típico continuo (ej: 18 a 75 años)
df_ocupados = df[(df['activ'] == 1.0) & (df['yoprcor'] > 0) & (df['edad'] >= 18) & (df['edad'] <= 75)].copy()

# Calculamos el ingreso promedio exacto por edad y zona (area)
# area: 1 es Urbano, 2 es Rural
df_brecha = df_ocupados.groupby(['edad', 'area'], observed=True)['yoprcor'].mean().unstack()
df_brecha.columns = ['Urbano', 'Rural']
df_brecha = df_brecha.reset_index()

# Suavizado de datos con media móvil para una curva más estética
df_brecha['Urbano_suave'] = df_brecha['Urbano'].rolling(window=3, center=True, min_periods=1).mean()
df_brecha['Rural_suave'] = df_brecha['Rural'].rolling(window=3, center=True, min_periods=1).mean()

# 2. Configuración de diseño Premium (Aesthetics)
color_urbano = '#1F4E5B' # Deep Teal / Azul Océano elegante
color_rural = '#D97706'  # Naranja Cálido / Ocre Tierra sofisticado

# 3. Creación del gráfico
fig = go.Figure()

# Línea de Rural
fig.add_trace(go.Scatter(
    x=df_brecha['edad'],
    y=df_brecha['Rural_suave'],
    name='Rural',
    mode='lines',
    line=dict(color=color_rural, width=4.5, shape='spline', smoothing=1.3),
    hovertemplate='<b>Edad:</b> %{x} años<br><b>Ingreso Promedio:</b> $%{y:,.0f}<extra></extra>'
))

# Línea de Urbano con sombreado hacia Rural
fig.add_trace(go.Scatter(
    x=df_brecha['edad'],
    y=df_brecha['Urbano_suave'],
    name='Urbano',
    mode='lines',
    line=dict(color=color_urbano, width=4.5, shape='spline', smoothing=1.3),
    fill='tonexty',
    fillcolor='rgba(217, 119, 6, 0.12)',
    hovertemplate='<b>Edad:</b> %{x} años<br><b>Ingreso Promedio:</b> $%{y:,.0f}<extra></extra>'
))

# 4. Ajustes de Layout (Typography, Grids, Backgrounds)
fig.update_layout(
    title=dict(
        text='<b>Brecha de Ingresos por Zona Geográfica y Edad</b>',
        font=dict(family='Inter, Roboto, Arial, sans-serif', size=28, color='#1A202C'),
        x=0.02,
        y=0.96
    ),
    xaxis=dict(
        title=dict(
            text='<b>Edad (Años)</b>',
            font=dict(family='Inter, Arial', size=18, color='#2D3748')
        ),
        showgrid=False,
        showline=True,
        linecolor='rgba(0,0,0,0.3)',
        linewidth=2,
        tickfont=dict(family='Inter, Arial', size=15, color='#4A5568'),
        dtick=5,
        range=[18, 75]
    ),
    yaxis=dict(
        title=dict(
            text='<b>Ingreso Promedio (CLP)</b>',
            font=dict(family='Inter, Arial', size=18, color='#2D3748')
        ),
        tickformat='$,.0f',
        gridcolor='rgba(0, 0, 0, 0.08)',
        gridwidth=1.5,
        griddash='dot',
        zeroline=False,
        showline=False,
        tickfont=dict(family='Inter, Arial', size=15, color='#4A5568')
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    hovermode='x unified',
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.05,
        xanchor='right',
        x=0.98,
        font=dict(family='Inter, Arial', size=16, color='#2D3748'),
        bgcolor='rgba(255, 255, 255, 0)',
        itemwidth=40
    ),
    height=650,
    margin=dict(t=120, b=80, l=90, r=40),
    hoverlabel=dict(
        bgcolor="white",
        font_size=15,
        font_family="Inter"
    )
)

fig.show()